In [9]:
import pandas as pd
import math

df = pd.read_csv("teen.csv")

target = df.columns[-1]
attributes = list(df.columns[:-1])

n = int(0.8*len(df))

train = df.iloc[:n]
test = df.iloc[n:]

print("Target:", target)
print("Training rows:", len(train))
print("Testing rows:", len(test))

Target: depression_label
Training rows: 960
Testing rows: 240


In [10]:
def train_naive_bayes(data):
    model = {}
    classes = data[target].unique()
    for c in classes:
        subset = data[data[target] == c]
        model[c] = {
            "prior": len(subset) / len(data),
            "likelihood": {}
        }
        for attr in attributes:
            model[c]["likelihood"][attr] = {}
            values = data[attr].unique()
            total = len(subset)
            for value in values:
                count = len(subset[subset[attr] == value])
                prob = (count + 1) / (total + len(values))
                model[c]["likelihood"][attr][value] = prob

    return model

In [11]:
def predict(model, row):

    best_class = None
    best_prob = -float("inf")

    for c in model:
        prob = math.log(model[c]["prior"])
        for attr in attributes:
            value = row[attr]
            if value in model[c]["likelihood"][attr]:
                prob += math.log(
                    model[c]["likelihood"][attr][value]
                )
        if prob > best_prob:
            best_prob = prob
            best_class = c
    return best_class

In [12]:
def accuracy(actual, predicted):
    return sum(a == p for a, p in zip(actual, predicted)) / len(actual)
model = train_naive_bayes(train)

actual = list(test[target])

predicted = [
    predict(model, row)
    for _, row in test.iterrows()
]

print("Accuracy:", accuracy(actual, predicted) * 100)

Accuracy: 96.66666666666667
